# Project Cycle 2: Confidence Intervals and One-Sample Inference
**Team Name:** [第7組]
**Team Members:** [113370229黃睿恩, 111370234黃柏維, 111A50012楊書雅]

## Step 1.1: Introduction & Research Questions
本專案旨在利用 YRBS 2007 資料集進行單一樣本推論。我們挑選了以下兩個變數進行分析：


1. **比例分析 (行為變數)**：`EverCigaretteUse` (是否曾吸菸)。
   * **基準值**：$p_0 = 0.50$
   * **成功定義**：原始編碼 1 (曾吸菸)。
   * **失敗定義**：原始編碼 2 (未曾吸菸)。
   * **研究問題**：美國青少年曾吸菸的母體比例，是否顯著不等於 0.50？
   * **缺失/無效值處理**：透過列表刪除法處理。該欄位中任何缺失值的記錄皆不納入最終分析。
   
2. **平均數分析 (連續變數)**：`BMIPCT` (BMI 百分位數)。
   * **基準值**：$\mu_0 = 65.0$
   * **研究問題**：美國青少年的平均 BMI 百分位數 (BMIPCT) 是否顯著不同於基準值 $\mu_0 = 65.0$？
   * **有效代碼**：從 `0.0` 到 `100.0` 的連續數值。
   * **缺失/無效值處理**：透過列表刪除法處理。缺失 BMI 數據的記錄將被移除，以確保兩個變數使用一致的樣本。

---
This project aims to conduct **one-sample inference** using the **YRBS 2007** dataset. We have selected the following two variables for analysis:

1. **Proportion Analysis (Behavioral Variable)**
* **Variable**: `EverCigaretteUse` (Lifetime cigarette use)
* **Baseline Value**: $p_0 = 0.50$
* **Operational Definitions**:
    * **Success**: Original code 1 (Ever smoked)
    * **Failure**: Original code 2 (Never smoked)
* **Research Question**: Is the population proportion of U.S. adolescents who have ever smoked significantly different from 0.50?
* **Missing/Invalid Values**: Handled via listwise deletion. Any record with a missing value in this column is excluded from the final analysis.

2. **Mean Analysis (Continuous Variable)**
* **Variable**: `BMIPCT` (Body Mass Index Percentile)
* **Baseline Value**: $\mu_0 = 65.0$
* **Research Question**: Is the mean BMI percentile (BMIPCT) of U.S. adolescents significantly different from the hypothesized value of $\mu_0 = 65.0$?
* **Valid Codes**: Continuous values ranging from `0.0` to `100.0`.
* **Missing/Invalid Values**: Handled via listwise deletion. Records with missing BMI data are removed to maintain a consistent sample for both variables.

## 1.2 Setup & Data Loading

In [3]:
import pandas as pd
import numpy as np
import os

# 1. 自動建立資料夾結構 (確保儲存時不會報錯) [cite: 140-151]
folders = ['../data/processed', '../outputs/figures', '../outputs/tables']
for folder in folders:
    if not os.path.exists(folder):
        os.makedirs(folder)

# 2. 載入原始資料 [cite: 166]
# 請確保 YRBS_2007.csv 放在 project-cycle-2/data/raw/ 底下
df_raw = pd.read_csv('../data/raw/YRBS_2007.csv')

# 3. 變數定義與缺失值回報 [cite: 63-70]
# 選擇變數：是否曾吸菸 (EverCigaretteUse) 與 BMI 百分位 (BMIPCT)
df_selected = df_raw[['EverCigaretteUse', 'BMIPCT']].copy()

# 依照要求添加：顯示各欄位缺失值數量
print("各欄位缺失值數量：")
print(df_selected.isnull().sum())

# 4. 執行資料清理 (移除缺失值)
df_cleaned = df_selected.dropna()

# 儲存第一份處理檔案：yrbs_cleaned.csv
df_cleaned.to_csv('../data/processed/yrbs_cleaned.csv', index=False)

# 5. 執行重編碼規則 (Recoding Rules) [cite: 33-35]
# EverCigaretteUse: 原始代碼 1 = 曾吸菸 (Success), 2 = 未曾吸菸 (Failure)
# 為方便計算比例，我們將 1 保持為 1，將 2 改為 0
df_recoded = df_cleaned.copy()
df_recoded['EverCig_Binary'] = np.where(df_recoded['EverCigaretteUse'] == 1, 1, 0)

# 儲存第二份處理檔案：yrbs_recoded.csv
df_recoded.to_csv('../data/processed/yrbs_recoded.csv', index=False)

# 6. 回報最終處理結果 [cite: 70]
print("-" * 30)
print(f"原始資料樣本數: {len(df_raw)}")
print(f"清理後樣本數 (已移除缺失值): {len(df_cleaned)}")
print(f"重編碼後最終樣本數: {len(df_recoded)}")
print("-" * 30)
print("檔案已成功儲存至 data/processed/ 資料夾。")

各欄位缺失值數量：
EverCigaretteUse    440
BMIPCT              979
dtype: int64
------------------------------
原始資料樣本數: 14041
清理後樣本數 (已移除缺失值): 12683
重編碼後最終樣本數: 12683
------------------------------
檔案已成功儲存至 data/processed/ 資料夾。


## 1.3 References Setup 參考資料設定

According to the project structure, the `references/` folder stores the "metadata" of our analysis. This includes the definitions of variables and the logic behind recoding. We will now programmatically generate these documents.

根據專案結構，`references/` 資料夾存放的是我們分析的「中繼資料」。這包括變數的定義以及重編碼的邏輯。我們現在將透過程式自動生成這些文件。

In [4]:
import os

# 1. 設定並建立 references 資料夾
ref_dir = '../references'
os.makedirs(ref_dir, exist_ok=True)

# 2. 自動偵測目前的有效樣本量 (n)
# 依序檢查 df_recoded 或 df_cleaned 是否存在於目前的命名空間
df_to_check = ['df_recoded', 'df_cleaned']
current_n = "N/A (Please check dataframe name)"

for name in df_to_check:
    if name in globals():
        current_n = len(globals()[name])
        break

# 3. 定義 variable_definitions.md 內容
var_defs = """# Variable Definitions (變數定義)

This file documents the original variables from the YRBS 2007 dataset used in this analysis.

| Original Code | Variable Name | Description | Data Type |
| :--- | :--- | :--- | :--- |
| Q31 | EverCigaretteUse | Has the respondent ever tried cigarette smoking, even one or two puffs? | Categorical |
| Q2 | BMIPCT | BMI percentile calculated based on age, sex, height, and weight. | Continuous |

* **Source**: Centers for Disease Control and Prevention (CDC) - 2007 YRBS.
"""

# 4. 定義 recoding_rules.md 內容
# 注意：使用 fr (f-string + raw string) 來同時支援變數注入與 LaTeX 反斜線
recode_rules = fr"""# Recoding Rules and Benchmarks (重編碼規則與基準值)

This file documents the transformations applied to the raw data for statistical inference.

### 1. Behavioral Variable: Cigarette Use
* **Transformation**: Binary recoding.
* **Logic**: 
    * Raw value `1` (Yes) -> `1` (Success)
    * Raw value `2` (No) -> `0` (Failure)
* **Neutral Benchmark ($p_0$)**: 0.50 (To test if more than half of the population experimented with smoking).

### 2. Continuous Variable: BMI Percentile
* **Transformation**: None (Used raw percentile values).
* **Missing Values**: Applied **Listwise Deletion** to ensure consistent sample size ($n = {current_n}$).
* **Historical Benchmark ($\mu_0$)**: 65.0 (Based on historical average standards).

### 3. Data Cleaning
* **Process**: Removed all records with missing values in either target variable to maintain analysis integrity.
"""

# 5. 執行檔案寫入
files_to_write = {
    'variable_definitions.md': var_defs,
    'recoding_rules.md': recode_rules
}

try:
    for filename, content in files_to_write.items():
        file_path = os.path.join(ref_dir, filename)
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
    
    print(f"✨ 成功產生日誌檔案！")
    print(f"路徑位置: {os.path.abspath(ref_dir)}")
except Exception as e:
    print(f"❌ 寫入過程中發生錯誤: {e}")

✨ 成功產生日誌檔案！
路徑位置: c:\Users\user\Desktop\project-cycle-2-final\references


**Conclusion:**
We have successfully created the `references/` folder and populated it with `variable_definitions.md` and `recoding_rules.md`. These files serve as the "Data Dictionary" for our project, ensuring that any other researcher can understand how our variables were defined and transformed.

**結論：**
我們已成功建立 `references/` 資料夾，並生成了 `variable_definitions.md` 與 `recoding_rules.md`。這些檔案充當了我們專案的「數據字典」，確保任何其他研究人員都能理解我們的變數是如何定義與轉換的。